In [137]:
import pandas as pd
from pathlib import Path

import re

In [138]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [139]:
INPUT_FILE = Path(r"C:\Users\user\Downloads\Data Samples\Data Samples\latest_monthly_orders_combined2.csv")
OUTPUT_DIR = Path(r"C:\Users\user\Downloads\Data Samples\Data Samples")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MART_DIR = OUTPUT_DIR / "Data_Marts"
MART_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_FILE)
print("Output:", OUTPUT_DIR)

Input: C:\Users\user\Downloads\Data Samples\Data Samples\latest_monthly_orders_combined2.csv
Output: C:\Users\user\Downloads\Data Samples\Data Samples


In [140]:

df = pd.read_csv(INPUT_FILE)

print("Raw shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
df.head()

Raw shape: (9994, 23)
Columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Source_File', 'Business_Month']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Source_File,Business_Month
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01


In [141]:
# Duplicate Row ID check
print("Duplicate Row IDs:", df["Row ID"].duplicated().sum())

# Null counts
null_report = (
    df.isna()
      .sum()
      .reset_index()
      .rename(columns={"index": "Column", 0: "Null Count"})
)

null_report["Null %"] = (null_report["Null Count"] / len(df) * 100).round(2)
print(null_report[null_report["Null Count"] > 0])


Duplicate Row IDs: 0
    Column  Null Count  Null %
20  Profit         234    2.34


In [142]:
# Blank-string check
blank_counts = {}

for col in df.columns:
    blank_counts[col] = (df[col].astype("string").str.strip() == "").sum()

blank_report = pd.Series(blank_counts, name="Blank Count")
print(blank_report[blank_report > 0])


Series([], Name: Blank Count, dtype: int64)


In [143]:
# A. Missing Profit
missing_profit_mask = df["Profit"].isna()
print("Missing Profit rows:", int(missing_profit_mask.sum()))

# B. Negative Quantity
negative_quantity_mask = df["Quantity"] < 0
print("Negative Quantity rows:", int(negative_quantity_mask.sum()))

# C. Customer IDs associated with multiple states
state_check = df.groupby("Customer ID")["State"].nunique(dropna=True)
multi_state_customer_ids = state_check[state_check > 1].index

customer_multiple_state_mask = df["Customer ID"].isin(multi_state_customer_ids)

print("Affected Customer IDs:", len(multi_state_customer_ids))
print("Affected Row IDs:", df.loc[customer_multiple_state_mask, "Row ID"].nunique())


Missing Profit rows: 234
Negative Quantity rows: 4
Affected Customer IDs: 778
Affected Row IDs: 9948


In [144]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Source_File,Business_Month
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,201901_Orders_2019_02_04_15_41_32.csv,2019-01-01


In [145]:
df.shape

(9994, 23)

In [146]:
df = df.drop_duplicates()

In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          9994 non-null   int64  
 1   Order ID        9994 non-null   object 
 2   Order Date      9994 non-null   object 
 3   Ship Date       9994 non-null   object 
 4   Ship Mode       9994 non-null   object 
 5   Customer ID     9994 non-null   object 
 6   Customer Name   9994 non-null   object 
 7   Segment         9994 non-null   object 
 8   Country         9994 non-null   object 
 9   City            9994 non-null   object 
 10  State           9994 non-null   object 
 11  Postal Code     9994 non-null   int64  
 12  Region          9994 non-null   object 
 13  Product ID      9994 non-null   object 
 14  Category        9994 non-null   object 
 15  Sub-Category    9994 non-null   object 
 16  Product Name    9994 non-null   object 
 17  Sales           9994 non-null   f

In [148]:
#1 data incosistency


df.isna().sum()

#there are 234 nul values in profit


Row ID              0
Order ID            0
Order Date          0
Ship Date           0
Ship Mode           0
Customer ID         0
Customer Name       0
Segment             0
Country             0
City                0
State               0
Postal Code         0
Region              0
Product ID          0
Category            0
Sub-Category        0
Product Name        0
Sales               0
Quantity            0
Discount            0
Profit            234
Source_File         0
Business_Month      0
dtype: int64

In [149]:
columns = ['Order ID', 'Customer ID', 'Product ID', 
           'Order Date', 'Ship Date', 'Sales']

print(df[columns].isna().sum())

Order ID       0
Customer ID    0
Product ID     0
Order Date     0
Ship Date      0
Sales          0
dtype: int64


In [150]:
for col in df.columns:
    count = (df[col].astype(str).str.strip() == '').sum()
    print(col, count)


#to make sure no column have an empty string

Row ID 0
Order ID 0
Order Date 0
Ship Date 0
Ship Mode 0
Customer ID 0
Customer Name 0
Segment 0
Country 0
City 0
State 0
Postal Code 0
Region 0
Product ID 0
Category 0
Sub-Category 0
Product Name 0
Sales 0
Quantity 0
Discount 0
Profit 0
Source_File 0
Business_Month 0


In [151]:
print("\n Duplicate Row IDs")
print(df['Row ID'].duplicated().sum())



 Duplicate Row IDs
0


In [152]:
# D. Product IDs associated with multiple Product Names
product_name_check = (
    df.groupby("Product ID")["Product Name"]
      .nunique(dropna=True)
)

conflicting_product_ids = product_name_check[product_name_check > 1].index
product_conflict_mask = df["Product ID"].isin(conflicting_product_ids)

print("Affected Product IDs:", len(conflicting_product_ids))
print("Affected rows:", int(product_conflict_mask.sum()))
print("Conflicting Product IDs:")
print(list(conflicting_product_ids))

Affected Product IDs: 5
Affected rows: 54
Conflicting Product IDs:
['FUR-BO-10002213', 'FUR-CH-10001146', 'FUR-FU-10001473', 'FUR-FU-10004017', 'FUR-FU-10004091']


In [153]:
#2 data incosistency

print("\n--- Negative/Invalid Values ---")
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("Sales < 0:", (df['Sales'] < 0).sum())
print("Discount < 0:", (df['Discount'] < 0).sum())
print("Discount > 1:", (df['Discount'] > 1).sum())

#there are 4 negative values in quantity


--- Negative/Invalid Values ---
Quantity <= 0: 4
Sales < 0: 0
Discount < 0: 0
Discount > 1: 0


In [154]:

print("\n Ship Mode Values")
print(df['Ship Mode'].value_counts(dropna=False))

#no diference in capatilization between unique values


 Ship Mode Values
Ship Mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64


In [155]:
print("\n Segment Values")
print(df['Segment'].value_counts(dropna=False))


 Segment Values
Segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64


In [156]:
postal_conflicts = postal_conflicts[
    postal_conflicts > 1
]
print(postal_conflicts)

Postal Code
92024    2
Name: City, dtype: int64


In [157]:

#3 data incosistency



postal_conflicts = (
    df.groupby('Postal Code')['City']
      .nunique()
)

conflict_codes = postal_conflicts[postal_conflicts > 1].index

problematic_rows = df[
    df['Postal Code'].isin(conflict_codes)
].sort_values('Postal Code')

print(problematic_rows[['Row ID', 'Postal Code', 'City']])


# the post cose associated twice with 2 cities, equires validation/SME clarification

      Row ID  Postal Code       City
85      1544        92024  San Diego
4628    2821        92024  Encinitas
4943    1552        92024  San Diego
4985    3529        92024  San Diego
6123    7190        92024  San Diego
6124    7191        92024  San Diego
6155    4397        92024  San Diego
6540    8685        92024  San Diego
4627    2820        92024  Encinitas
8017    6699        92024  San Diego
8027    6701        92024  San Diego
9608    2675        92024  San Diego
9649    1925        92024  San Diego
9779    2949        92024  San Diego
9781    2947        92024  San Diego
9791    2948        92024  San Diego
9834    1114        92024  Encinitas
8018    6700        92024  San Diego
9835    1115        92024  Encinitas
4289    2607        92024  San Diego
3644    9062        92024  San Diego
86      1545        92024  San Diego
861      596        92024  San Diego
862      597        92024  San Diego
864      598        92024  San Diego
867      595        92024  San Diego
1

In [158]:
#4 data incosistency

# to check if one customer can be associated with more than one state

state_check = df.groupby('Customer ID')['State'].nunique()
print((state_check > 1).sum())

#the location of a customer can change over time , so we may have one order ID 
#of a customer who used to live in dfferent state, to solve it we may spot the order
#id with different state rows, and adopt the state from most updated recods, but
#we need SME validation for this due to the large number of rows and to
#determine whether it's a real business inconsistency

#unresolvable inconsistency

778


In [159]:
##5 data incosistency
#check if the order date is before dilevery date

print("Order Date after Ship Date:",
      (df['Order Date'] > df['Ship Date']).sum())

#the outcome is unrelaistic, may indicate unified standardization of date format 

Order Date after Ship Date: 1235


In [160]:
print(df['Order Date'].dropna().unique()[:100])


#'08-01-2019' this date formate is different than all other values using DD-MM-YYYY format

['04-01-2019' '05-01-2019' '06-01-2019' '07-01-2019' '08-01-2019'
 '10-01-2019' '11-01-2019' '12-01-2019' '14-01-2019' '1/15/2019'
 '16-01-2019' '17-01-2019' '19-01-2019' '20-01-2019' '21-01-2019'
 '22-01-2019' '24-01-2019' '27-01-2019' '28-01-2019' '29-01-2019'
 '31-01-2019' '01-02-2019' '02-02-2019' '03-02-2019' '04-02-2019'
 '05-02-2019' '07-02-2019' '08-02-2019' '09-02-2019' '12-02-2019'
 '13-02-2019' '15-02-2019' '16-02-2019' '17-02-2019' '18-02-2019'
 '19-02-2019' '21-02-2019' '22-02-2019' '23-02-2019' '24-02-2019'
 '25-02-2019' '28-02-2019' '01-03-2019' '02-03-2019' '03-03-2019'
 '04-03-2019' '05-03-2019' '07-03-2019' '10-03-2019' '11-03-2019'
 '14-03-2019' '15-03-2019' '16-03-2019' '17-03-2019' '18-03-2019'
 '19-03-2019' '21-03-2019' '22-03-2019' '23-03-2019' '24-03-2019'
 '25-03-2019' '26-03-2019' '28-03-2019' '29-03-2019' '30-03-2019'
 '31-03-2019' '01-04-2019' '02-04-2019' '03-04-2019' '04-04-2019'
 '05-04-2019' '06-04-2019' '07-04-2019' '08-04-2019' '11-04-2019'
 '12-04-201

In [161]:
print(df['Ship Date'].dropna().unique()[:100])


['08-01-2019' '09-01-2019' '13-01-2019' '11-01-2019' '14-01-2019'
 '16-01-2019' '15-01-2019' '19-01-2019' '17-01-2019' '18-01-2019'
 '22-01-2019' '21-01-2019' '27-01-2019' '26-01-2019' '24-01-2019'
 '28-01-2019' '29-01-2019' '01-02-2019' '30-01-2019' '03-02-2019'
 '04-02-2019' '05-02-2019' '10-02-2019' '07-02-2019' '09-02-2019'
 '11-02-2019' '13-02-2019' '16-02-2019' '19-02-2019' '20-02-2019'
 '22-02-2019' '25-02-2019' '27-02-2019' '26-02-2019' '28-02-2019'
 '02-03-2019' '03-03-2019' '05-03-2019' '06-03-2019' '07-03-2019'
 '08-03-2019' '09-03-2019' '04-03-2019' '12-03-2019' '11-03-2019'
 '17-03-2019' '14-03-2019' '15-03-2019' '13-03-2019' '16-03-2019'
 '19-03-2019' '18-03-2019' '21-03-2019' '20-03-2019' '24-03-2019'
 '23-03-2019' '25-03-2019' '26-03-2019' '22-03-2019' '29-03-2019'
 '28-03-2019' '01-04-2019' '30-03-2019' '31-03-2019' '02-04-2019'
 '03-04-2019' '05-04-2019' '04-04-2019' '06-04-2019' '08-04-2019'
 '09-04-2019' '07-04-2019' '10-04-2019' '12-04-2019' '15-04-2019'
 '13-04-20

In [162]:
print(df['Ship Date'].astype(str).str.contains('/').sum())

0


In [163]:

print(df[df['Order Date'].astype(str).str.contains('/')][
    ['Row ID', 'Order Date', 'Ship Date']
])

      Row ID Order Date   Ship Date
32      6475  1/15/2019  16-01-2019
4129    5968  1/15/2021  17-01-2021
4130    2878  1/15/2021  19-01-2021
4131    3236  1/15/2021  15-01-2021
4132    2880  1/15/2021  19-01-2021
4133    2879  1/15/2021  19-01-2021
4134    3744  1/15/2021  21-01-2021
4135    3742  1/15/2021  21-01-2021
4136    3743  1/15/2021  21-01-2021
6720    1183  1/15/2022  16-01-2022
6721    9863  1/15/2022  21-01-2022
6722    9862  1/15/2022  21-01-2022
6723    4914  1/15/2022  19-01-2022
6724    4913  1/15/2022  19-01-2022
6725    4912  1/15/2022  19-01-2022
6726    4911  1/15/2022  19-01-2022
6727    9865  1/15/2022  21-01-2022
6728    9864  1/15/2022  21-01-2022
6729    1184  1/15/2022  16-01-2022
6730    9866  1/15/2022  21-01-2022
6731    4910  1/15/2022  19-01-2022
6732    6980  1/15/2022  20-01-2022
6733    4165  1/15/2022  17-01-2022
6734    4166  1/15/2022  17-01-2022
6735    9861  1/15/2022  21-01-2022
6736    9859  1/15/2022  21-01-2022
6737    9860  1/15/2022  21-

In [164]:
#to fix the date error, firt thing is to standardize the 27 in 
# order date column,

df['Order Date'] = pd.to_datetime(
    df['Order Date'],
    format='mixed',
    dayfirst=True
)

df['Ship Date'] = pd.to_datetime(
    df['Ship Date'],
    dayfirst=True
)

In [165]:
print("Order Date after Ship Date:",
      (df['Order Date'] > df['Ship Date']).sum())

#problem solved

Order Date after Ship Date: 0


In [166]:
#check If each Order ID always have only one unique Customer Name

df.groupby("Order ID")["Customer Name"].nunique().loc[lambda x: x > 1]

Series([], Name: Customer Name, dtype: int64)

In [167]:
# A. Missing Profit
missing_profit_mask = df["Profit"].isna()
print("Missing Profit rows:", int(missing_profit_mask.sum()))

# B. Negative Quantity
negative_quantity_mask = df["Quantity"] < 0
print("Negative Quantity rows:", int(negative_quantity_mask.sum()))

# C. Customer IDs associated with multiple states
state_check = df.groupby("Customer ID")["State"].nunique(dropna=True)
multi_state_customer_ids = state_check[state_check > 1].index

customer_multiple_state_mask = df["Customer ID"].isin(multi_state_customer_ids)

print("Affected Customer IDs:", len(multi_state_customer_ids))
print("Affected Row IDs:", df.loc[customer_multiple_state_mask, "Row ID"].nunique())


Missing Profit rows: 234
Negative Quantity rows: 4
Affected Customer IDs: 778
Affected Row IDs: 9948


In [168]:
# D. Product IDs associated with multiple Product Names
product_name_check = (
    df.groupby("Product ID")["Product Name"]
      .nunique(dropna=True)
)

conflicting_product_ids = product_name_check[product_name_check > 1].index
product_conflict_mask = df["Product ID"].isin(conflicting_product_ids)

print("Affected Product IDs:", len(conflicting_product_ids))
print("Affected rows:", int(product_conflict_mask.sum()))
print("Conflicting Product IDs:")
print(list(conflicting_product_ids))


Affected Product IDs: 5
Affected rows: 54
Conflicting Product IDs:
['FUR-BO-10002213', 'FUR-CH-10001146', 'FUR-FU-10001473', 'FUR-FU-10004017', 'FUR-FU-10004091']


In [169]:
# E. Inconsistent Order Date formatting
# The dominant source format is DD-MM-YYYY.
# The known non-dominant formats are slash-formatted dates and ISO dates.

order_date_raw = df["Order Date"].astype("string").str.strip()

slash_date_mask = order_date_raw.str.contains(r"/", regex=True, na=False)
iso_date_mask = order_date_raw.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)

date_format_mask = slash_date_mask | iso_date_mask

print("Slash-formatted Order Date rows:", int(slash_date_mask.sum()))
print("ISO-formatted Order Date rows:", int(iso_date_mask.sum()))
print("Inconsistent Order Date Format rows:", int(date_format_mask.sum()))


Slash-formatted Order Date rows: 0
ISO-formatted Order Date rows: 9994
Inconsistent Order Date Format rows: 9994


In [170]:
# F. Check for invalid business ordering of dates.
# Dates are parsed only for validation here; the actual cleaning happens below.

order_dates_check = pd.to_datetime(
    df["Order Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

ship_dates_check = pd.to_datetime(
    df["Ship Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print(
    "Order Date after Ship Date:",
    int((order_dates_check > ship_dates_check).sum())
)


Order Date after Ship Date: 0


In [172]:
# Two examples for each issue

examples = {}

examples["Missing Profit"] = (
    df.loc[missing_profit_mask, ["Row ID", "Profit"]]
      .head(2)
)

examples["Negative Quantity"] = (
    df.loc[negative_quantity_mask, ["Row ID", "Quantity"]]
      .head(2)
)

examples["Inconsistent Order Date Format"] = (
    df.loc[date_format_mask, ["Row ID", "Order Date", "Ship Date"]]
      .head(2)
)

examples["Product ID with Multiple Product Names"] = (
    df.loc[product_conflict_mask, ["Row ID", "Product ID", "Product Name"]]
      .drop_duplicates()
      .sort_values(["Product ID", "Row ID"])
      .head(2)
)

examples["Customer ID with Multiple States"] = (
    df.loc[customer_multiple_state_mask, ["Row ID", "Customer ID", "Customer Name", "State"]]
      .drop_duplicates()
      .sort_values(["Customer ID", "Row ID"])
      .head(2)
)

for issue, example_df in examples.items():
    print("\n---", issue, "---")
    display(example_df)



--- Missing Profit ---


,Row ID,Profit
7627,9951,NaN
7628,9949,NaN



--- Negative Quantity ---


,Row ID,Quantity
819,1838,-3
2837,1505,-4



--- Inconsistent Order Date Format ---


,Row ID,Order Date,Ship Date
0,7981,2019-01-04,2019-01-08
1,740,2019-01-05,2019-01-09



--- Product ID with Multiple Product Names ---


,Row ID,Product ID,Product Name
3451,2116,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases
4145,2472,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish"



--- Customer ID with Multiple States ---


,Row ID,Customer ID,Customer Name,State
7853,1160,AA-10315,Alex Avila,Minnesota
7855,1161,AA-10315,Alex Avila,Minnesota


In [173]:
clean_df = df.copy()

# Standardize text
text_columns = [
    "Order ID", "Customer ID", "Customer Name", "Segment",
    "Country", "City", "State", "Region",
    "Product ID", "Product Name", "Category", "Sub-Category",
    "Ship Mode"
]

for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Standardize dates
clean_df["Order Date"] = pd.to_datetime(
    clean_df["Order Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

clean_df["Ship Date"] = pd.to_datetime(
    clean_df["Ship Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Standardize numeric fields
numeric_columns = [
    "Row ID", "Postal Code", "Sales", "Quantity", "Discount", "Profit"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

print("Rows after technical standardization:", len(clean_df))


Rows after technical standardization: 9994


In [174]:
# Recalculate masks from the standardized dataframe
missing_profit_mask = clean_df["Profit"].isna()
negative_quantity_mask = clean_df["Quantity"] < 0

product_name_check = (
    clean_df.groupby("Product ID")["Product Name"]
            .nunique(dropna=True)
)
conflicting_product_ids = product_name_check[product_name_check > 1].index
product_conflict_mask = clean_df["Product ID"].isin(conflicting_product_ids)

# Customer multiple-state records are deliberately NOT quarantined.
state_check = clean_df.groupby("Customer ID")["State"].nunique(dropna=True)
multi_state_customer_ids = state_check[state_check > 1].index
customer_multiple_state_mask = clean_df["Customer ID"].isin(multi_state_customer_ids)

quarantine_mask = (
    missing_profit_mask
    | negative_quantity_mask
    | product_conflict_mask
)

quarantine_df = clean_df.loc[quarantine_mask].copy()
warehouse_df = clean_df.loc[~quarantine_mask].copy()

print("Missing Profit:", int(missing_profit_mask.sum()))
print("Negative Quantity:", int(negative_quantity_mask.sum()))
print("Product conflict:", int(product_conflict_mask.sum()))
print("Customer multiple states:", int(customer_multiple_state_mask.sum()))
print("Quarantined rows:", len(quarantine_df))
print("Final warehouse rows:", len(warehouse_df))



Missing Profit: 234
Negative Quantity: 4
Product conflict: 54
Customer multiple states: 9948
Quarantined rows: 291
Final warehouse rows: 9703


In [175]:
# Confirm the expected overlap and final count.
overlap_missing_product = int((missing_profit_mask & product_conflict_mask).sum())

print("Missing Profit ∩ Product Conflict:", overlap_missing_product)
print("Expected quarantine calculation: 234 + 4 + 54 - 1 =", 234 + 4 + 54 - 1)
print("Actual quarantined rows:", len(quarantine_df))

assert overlap_missing_product == 1, "Expected one overlap between Missing Profit and Product conflict."
assert len(quarantine_df) == 291, "Expected 291 quarantined rows."
assert len(warehouse_df) == 9703, "Expected 9,703 final warehouse rows."


Missing Profit ∩ Product Conflict: 1
Expected quarantine calculation: 234 + 4 + 54 - 1 = 291
Actual quarantined rows: 291


In [176]:
warehouse_df = warehouse_df.rename(columns={
    "Product ID": "Product_ID",
    "Product Name": "Product_Name",
    "Customer ID": "Customer_ID",
    "Customer Name": "Customer_Name",
    "Sub-Category": "Sub_Category",
    "Postal Code": "Postal_Code",
    "Order Date": "Order_Date",
    "Ship Date": "Ship_Date",
    "Ship Mode": "Ship_Mode",
    "Row ID": "Row_ID"
})

print("Warehouse source rows:", len(warehouse_df))


Warehouse source rows: 9703


In [177]:
# DIM_CATEGORY
dim_category = (
    warehouse_df[["Category", "Sub_Category"]]
    .drop_duplicates()
    .sort_values(["Category", "Sub_Category"])
    .reset_index(drop=True)
)

dim_category.insert(0, "Category_Key", range(1, len(dim_category) + 1))


In [178]:
# DIM_PRODUCT
# Built from the cleaned data so ambiguous product IDs do not violate Product_ID uniqueness.

dim_product = (
    warehouse_df[["Product_ID", "Product_Name", "Category", "Sub_Category"]]
    .drop_duplicates()
    .merge(
        dim_category,
        on=["Category", "Sub_Category"],
        how="left"
    )
    [["Product_ID", "Product_Name", "Category_Key"]]
    .sort_values(["Product_ID", "Product_Name"])
    .reset_index(drop=True)
)

# A product ID must map to exactly one master record.
product_id_counts = dim_product.groupby("Product_ID").size()
assert (product_id_counts > 1).sum() == 0, "DIM_PRODUCT contains duplicate Product_ID values."

dim_product.insert(0, "Product_Key", range(1, len(dim_product) + 1))

print("DIM_PRODUCT rows:", len(dim_product))


DIM_PRODUCT rows: 1880


In [179]:
# DIM_LOCATION
# Location remains transaction-level so historical delivery geography is preserved.

dim_location = (
    warehouse_df[
        ["Country", "City", "State", "Postal_Code", "Region"]
    ]
    .drop_duplicates()
    .sort_values(["Country", "State", "City", "Postal_Code"])
    .reset_index(drop=True)
)

dim_location.insert(0, "Location_Key", range(1, len(dim_location) + 1))

print("DIM_LOCATION rows:", len(dim_location))


DIM_LOCATION rows: 627


In [180]:
# DIM_DATE
all_dates = pd.concat([
    warehouse_df["Order_Date"],
    warehouse_df["Ship_Date"]
]).dropna().drop_duplicates()

dim_date = pd.DataFrame({
    "Full_Date": pd.to_datetime(all_dates).dt.normalize()
})

dim_date = dim_date.sort_values("Full_Date").drop_duplicates().reset_index(drop=True)

dim_date.insert(
    0,
    "Date_Key",
    dim_date["Full_Date"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["Year"] = dim_date["Full_Date"].dt.year
dim_date["Quarter"] = dim_date["Full_Date"].dt.quarter
dim_date["Month"] = dim_date["Full_Date"].dt.month
dim_date["Month_Name"] = dim_date["Full_Date"].dt.month_name()

dim_date = dim_date[
    ["Date_Key", "Full_Date", "Year", "Quarter", "Month", "Month_Name"]
]

print("DIM_DATE rows:", len(dim_date))


DIM_DATE rows: 1411


In [181]:
# DIM_SEGMENT
dim_segment = (
    warehouse_df[["Segment"]]
    .drop_duplicates()
    .sort_values("Segment")
    .reset_index(drop=True)
)

dim_segment.insert(0, "Segment_Key", range(1, len(dim_segment) + 1))

print("DIM_SEGMENT rows:", len(dim_segment))


DIM_SEGMENT rows: 3


In [182]:
# DIM_CUSTOMER
# Use Segment_Key instead of repeating Segment text, matching the normalized model.

customer_base = (
    warehouse_df[["Customer_ID", "Customer_Name", "Segment"]]
    .drop_duplicates()
)

customer_segment_counts = customer_base.groupby("Customer_ID")["Segment"].nunique()
assert (customer_segment_counts > 1).sum() == 0, (
    "A Customer_ID maps to multiple segments in the cleaned source; SME review is required before "
    "creating a single customer master record."
)

dim_customer = (
    customer_base
    .merge(dim_segment, on="Segment", how="left")
    [["Customer_ID", "Customer_Name", "Segment_Key"]]
    .sort_values("Customer_ID")
    .reset_index(drop=True)
)

dim_customer.insert(0, "Customer_Key", range(1, len(dim_customer) + 1))

print("DIM_CUSTOMER rows:", len(dim_customer))


DIM_CUSTOMER rows: 793


In [183]:
fact = warehouse_df.copy()

# Connect Product
fact = fact.merge(
    dim_product[["Product_Key", "Product_ID"]],
    on="Product_ID",
    how="left",
    validate="many_to_one"
)

# Connect Customer
fact = fact.merge(
    dim_customer[["Customer_Key", "Customer_ID"]],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

# Connect Location
fact = fact.merge(
    dim_location[
        ["Location_Key", "Country", "City", "State", "Postal_Code", "Region"]
    ],
    on=["Country", "City", "State", "Postal_Code", "Region"],
    how="left",
    validate="many_to_one"
)

# Connect Order Date
fact["Order_Date"] = pd.to_datetime(fact["Order_Date"]).dt.normalize()

fact = fact.merge(
    dim_date[["Date_Key", "Full_Date"]],
    left_on="Order_Date",
    right_on="Full_Date",
    how="left",
    validate="many_to_one"
)

fact = fact.rename(columns={"Date_Key": "Order_Date_Key"})
fact = fact.drop(columns=["Full_Date"], errors="ignore")

# Connect Ship Date
fact["Ship_Date"] = pd.to_datetime(fact["Ship_Date"]).dt.normalize()

fact = fact.merge(
    dim_date[["Date_Key", "Full_Date"]],
    left_on="Ship_Date",
    right_on="Full_Date",
    how="left",
    validate="many_to_one"
)

fact = fact.rename(columns={"Date_Key": "Ship_Date_Key"})
fact = fact.drop(columns=["Full_Date"], errors="ignore")

# Connect Segment
fact = fact.merge(
    dim_segment,
    on="Segment",
    how="left",
    validate="many_to_one"
)

# Create Sales_Key
fact.insert(0, "Sales_Key", range(1, len(fact) + 1))

# Final FACT_SALES structure
fact = fact[
    [
        "Sales_Key",
        "Row_ID",
        "Order ID",
        "Product_Key",
        "Customer_Key",
        "Location_Key",
        "Order_Date_Key",
        "Ship_Date_Key",
        "Segment_Key",
        "Ship_Mode",
        "Sales",
        "Quantity",
        "Discount",
        "Profit"
    ]
]

print("FACT_SALES rows:", len(fact))
print("FACT_SALES columns:", fact.columns.tolist())


FACT_SALES rows: 9703
FACT_SALES columns: ['Sales_Key', 'Row_ID', 'Order ID', 'Product_Key', 'Customer_Key', 'Location_Key', 'Order_Date_Key', 'Ship_Date_Key', 'Segment_Key', 'Ship_Mode', 'Sales', 'Quantity', 'Discount', 'Profit']


In [184]:
# Primary-key uniqueness
assert dim_category["Category_Key"].is_unique
assert dim_product["Product_Key"].is_unique
assert dim_product["Product_ID"].is_unique
assert dim_location["Location_Key"].is_unique
assert dim_date["Date_Key"].is_unique
assert dim_segment["Segment_Key"].is_unique
assert dim_customer["Customer_Key"].is_unique
assert dim_customer["Customer_ID"].is_unique
assert fact["Sales_Key"].is_unique
assert fact["Row_ID"].is_unique

# Foreign-key completeness
assert fact["Product_Key"].notna().all()
assert fact["Customer_Key"].notna().all()
assert fact["Location_Key"].notna().all()
assert fact["Order_Date_Key"].notna().all()
assert fact["Ship_Date_Key"].notna().all()
assert fact["Segment_Key"].notna().all()

# Required business-quality checks
assert len(fact) == 9703
assert fact["Profit"].notna().all()
assert (fact["Quantity"] >= 0).all()

print("All key and referential-integrity checks passed.")


All key and referential-integrity checks passed.


In [185]:
marts = {
    "DIM_CATEGORY": dim_category,
    "DIM_PRODUCT": dim_product,
    "DIM_LOCATION": dim_location,
    "DIM_DATE": dim_date,
    "DIM_SEGMENT": dim_segment,
    "DIM_CUSTOMER": dim_customer,
    "FACT_SALES": fact
}

primary_keys = {
    "DIM_CATEGORY": "Category_Key",
    "DIM_PRODUCT": "Product_Key",
    "DIM_LOCATION": "Location_Key",
    "DIM_DATE": "Date_Key",
    "DIM_SEGMENT": "Segment_Key",
    "DIM_CUSTOMER": "Customer_Key",
    "FACT_SALES": "Sales_Key"
}

task6_rows = []

for name, mart in marts.items():
    pk = primary_keys[name]
    distinct_row_id = mart["Row_ID"].nunique() if "Row_ID" in mart.columns else ""

    task6_rows.append({
        "Data Mart System Name": name,
        "Count Rows": len(mart),
        "Count Distinct Primary Key": mart[pk].nunique(),
        "Count Distinct Row ID": distinct_row_id
    })

task6_rows = pd.DataFrame(task6_rows)

display(task6_rows)


,Data Mart System Name,Count Rows,Count Distinct Primary Key,Count Distinct Row ID
0,DIM_CATEGORY,17,17,
1,DIM_PRODUCT,1880,1880,
2,DIM_LOCATION,627,627,
3,DIM_DATE,1411,1411,
4,DIM_SEGMENT,3,3,
5,DIM_CUSTOMER,793,793,
6,FACT_SALES,9703,9703,9703


In [186]:
INPUT_FILE = Path(r"C:\Users\user\Downloads\Data Samples\Data Samples\latest_monthly_orders_combined2.csv")
OUTPUT_DIR = Path(r"C:\Users\user\Downloads\Data Samples\Data Samples")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MART_DIR = OUTPUT_DIR / "Data_Marts"
MART_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_FILE)
print("Output:", OUTPUT_DIR)

Input: C:\Users\user\Downloads\Data Samples\Data Samples\latest_monthly_orders_combined2.csv
Output: C:\Users\user\Downloads\Data Samples\Data Samples


In [187]:
print("DIM_DATE actual rows:", len(dim_date))
print("DIM_DATE distinct dates:", dim_date["Full_Date"].nunique())

DIM_DATE actual rows: 1411
DIM_DATE distinct dates: 1411


In [188]:
print(warehouse_df.columns.tolist())

['Row_ID', 'Order ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Source_File', 'Business_Month']


In [189]:
# ============================================================
# CREATE CLEANED WAREHOUSE DATA
# ============================================================

# 1. Find Product IDs that have multiple Product Names
product_check = (
    warehouse_df
    .groupby("Product_ID")["Product_Name"]
    .nunique()
)

conflicting_product_ids = product_check[
    product_check > 1
].index

print("Conflicting Product IDs:", len(conflicting_product_ids))
print("Product IDs:", conflicting_product_ids.tolist())


# 2. Create quarantine mask
#
# Quarantine:
# - Missing Profit
# - Negative Quantity
# - Product ID with multiple Product Names
#
# Do NOT quarantine date-format inconsistencies because
# valid dates will be standardized separately.
quarantine = (
    warehouse_df["Profit"].isna()
    | (warehouse_df["Quantity"] < 0)
    | warehouse_df["Product_ID"].isin(conflicting_product_ids)
)


# 3. Create cleaned warehouse dataset
warehouse_clean = warehouse_df[~quarantine].copy()


# 4. Validation
print("Original rows:", len(warehouse_df))
print("Quarantined rows:", quarantine.sum())
print("Clean rows:", len(warehouse_clean))

Conflicting Product IDs: 0
Product IDs: []
Original rows: 9703
Quarantined rows: 0
Clean rows: 9703


In [190]:
# ============================================================
# DIM_CATEGORY
# ============================================================

dim_category = (
    warehouse_clean[["Category", "Sub_Category"]]
    .drop_duplicates()
    .sort_values(["Category", "Sub_Category"])
    .reset_index(drop=True)
)

dim_category.insert(
    0,
    "Category_Key",
    range(1, len(dim_category) + 1)
)

print("DIM_CATEGORY rows:", len(dim_category))

DIM_CATEGORY rows: 17


In [191]:
# DIM_PRODUCT
# ============================================================

dim_product = (
    warehouse_clean[
        ["Product_ID", "Product_Name", "Category", "Sub_Category"]
    ]
    .drop_duplicates()
    .merge(
        dim_category,
        on=["Category", "Sub_Category"],
        how="left"
    )
    [
        ["Product_ID", "Product_Name", "Category_Key"]
    ]
    .sort_values(["Product_ID", "Product_Name"])
    .reset_index(drop=True)
)

# Verify Product_ID uniqueness
product_id_counts = dim_product.groupby("Product_ID").size()

assert (product_id_counts > 1).sum() == 0, (
    "DIM_PRODUCT contains duplicate Product_ID values."
)

dim_product.insert(
    0,
    "Product_Key",
    range(1, len(dim_product) + 1)
)

print("DIM_PRODUCT rows:", len(dim_product))
print("Distinct Product_ID:", dim_product["Product_ID"].nunique())

DIM_PRODUCT rows: 1880
Distinct Product_ID: 1880


In [192]:
# DIM_LOCATION
# ============================================================

dim_location = (
    warehouse_clean[
        ["Country", "City", "State", "Postal_Code", "Region"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Country", "State", "City", "Postal_Code"]
    )
    .reset_index(drop=True)
)

dim_location.insert(
    0,
    "Location_Key",
    range(1, len(dim_location) + 1)
)

print("DIM_LOCATION rows:", len(dim_location))

DIM_LOCATION rows: 627


In [193]:
# DIM_DATE
# ============================================================

# Standardize Order_Date and Ship_Date
warehouse_clean["Order_Date"] = pd.to_datetime(
    warehouse_clean["Order_Date"],
    errors="coerce"
)

warehouse_clean["Ship_Date"] = pd.to_datetime(
    warehouse_clean["Ship_Date"],
    errors="coerce"
)

all_dates = pd.concat([
    warehouse_clean["Order_Date"],
    warehouse_clean["Ship_Date"]
]).dropna()

dim_date = pd.DataFrame({
    "Full_Date": all_dates.dt.normalize().drop_duplicates()
})

dim_date = (
    dim_date
    .sort_values("Full_Date")
    .reset_index(drop=True)
)

dim_date.insert(
    0,
    "Date_Key",
    dim_date["Full_Date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["Year"] = dim_date["Full_Date"].dt.year
dim_date["Quarter"] = dim_date["Full_Date"].dt.quarter
dim_date["Month"] = dim_date["Full_Date"].dt.month
dim_date["Month_Name"] = dim_date["Full_Date"].dt.month_name()

dim_date = dim_date[
    [
        "Date_Key",
        "Full_Date",
        "Year",
        "Quarter",
        "Month",
        "Month_Name"
    ]
]

print("DIM_DATE rows:", len(dim_date))

DIM_DATE rows: 1411


In [194]:
# ============================================================
# DIM_SEGMENT
# ============================================================

dim_segment = (
    warehouse_clean[["Segment"]]
    .drop_duplicates()
    .sort_values("Segment")
    .reset_index(drop=True)
)

dim_segment.insert(
    0,
    "Segment_Key",
    range(1, len(dim_segment) + 1)
)

print("DIM_SEGMENT rows:", len(dim_segment))

DIM_SEGMENT rows: 3


In [195]:
# ============================================================
# DIM_CUSTOMER
# ============================================================

customer_base = (
    warehouse_clean[
        ["Customer_ID", "Customer_Name", "Segment"]
    ]
    .drop_duplicates()
)

customer_segment_counts = (
    customer_base
    .groupby("Customer_ID")["Segment"]
    .nunique()
)

assert (customer_segment_counts > 1).sum() == 0, (
    "A Customer_ID maps to multiple segments in the cleaned "
    "source; SME review is required."
)

dim_customer = (
    customer_base
    .merge(
        dim_segment,
        on="Segment",
        how="left"
    )
    [
        ["Customer_ID", "Customer_Name", "Segment_Key"]
    ]
    .sort_values("Customer_ID")
    .reset_index(drop=True)
)

dim_customer.insert(
    0,
    "Customer_Key",
    range(1, len(dim_customer) + 1)
)

print("DIM_CUSTOMER rows:", len(dim_customer))

DIM_CUSTOMER rows: 793


In [196]:
all_dates = pd.concat([
    warehouse_clean["Order_Date"],
    warehouse_clean["Ship_Date"]
]).dropna().drop_duplicates()

In [197]:
dim_date = pd.DataFrame({
    "Full_Date": pd.to_datetime(all_dates).dt.normalize()
})

In [198]:
print("DIM_DATE actual rows:", len(dim_date))

print("Distinct Order Dates:",
      warehouse_clean["Order_Date"].dropna().dt.normalize().nunique())

print("Distinct Ship Dates:",
      warehouse_clean["Ship_Date"].dropna().dt.normalize().nunique())

print("Combined distinct dates:",
      pd.concat([
          warehouse_clean["Order_Date"],
          warehouse_clean["Ship_Date"]
      ]).dropna().dt.normalize().nunique())

DIM_DATE actual rows: 1411
Distinct Order Dates: 1213
Distinct Ship Dates: 1309
Combined distinct dates: 1411


In [199]:
expected_counts = {
    "DIM_CATEGORY": 17,
    "DIM_PRODUCT": 1880,
    "DIM_LOCATION": 627,
    "DIM_DATE": 1411,
    "DIM_SEGMENT": 3,
    "DIM_CUSTOMER": 793,
    "FACT_SALES": 9703
}

actual_counts = dict(zip(
    task6_rows["Data Mart System Name"],
    task6_rows["Count Rows"]
))

for name, expected in expected_counts.items():
    print(f"{name}: expected={expected}, actual={actual_counts[name]}")
    assert actual_counts[name] == expected, (
        f"{name} count differs from the expected corrected target."
    )

assert fact["Sales_Key"].nunique() == 9703
assert fact["Row_ID"].nunique() == 9703

print("\nAll expected Task 6 counts passed.")

DIM_CATEGORY: expected=17, actual=17
DIM_PRODUCT: expected=1880, actual=1880
DIM_LOCATION: expected=627, actual=627
DIM_DATE: expected=1411, actual=1411
DIM_SEGMENT: expected=3, actual=3
DIM_CUSTOMER: expected=793, actual=793
FACT_SALES: expected=9703, actual=9703

All expected Task 6 counts passed.


In [200]:
for name, mart in marts.items():
    file_path = MART_DIR / f"{name}.csv"
    mart.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )
    print(f"Exported: {file_path}")


Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\DIM_CATEGORY.csv
Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\DIM_PRODUCT.csv
Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\DIM_LOCATION.csv
Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\DIM_DATE.csv
Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\DIM_SEGMENT.csv
Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\DIM_CUSTOMER.csv
Exported: C:\Users\user\Downloads\Data Samples\Data Samples\Data_Marts\FACT_SALES.csv


In [201]:
# Package all data-mart CSVs
task6_zip = OUTPUT_DIR / "Task_6_1_Data_Marts.zip"

with zipfile.ZipFile(task6_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for csv_file in sorted(MART_DIR.glob("*.csv")):
        archive.write(csv_file, arcname=csv_file.name)

print("Created:", task6_zip)

Created: C:\Users\user\Downloads\Data Samples\Data Samples\Task_6_1_Data_Marts.zip


In [205]:

task6_rows_file = OUTPUT_DIR / "Task_6_2_Data_Marts_Rows.csv"

task6_rows.to_csv(
    task6_rows_file,
    index=False,
    encoding="utf-8-sig"
)

print("File created:", task6_rows_file)


File created: C:\Users\user\Downloads\Data Samples\Data Samples\Task_6_2_Data_Marts_Rows.csv
